# Link Prediction avec GNN
Pipeline complet : chargement → construction du graphe PyG → entraînement → prédiction.

In [1]:
import pandas as pd
import numpy as np
import networkx as nx
import torch
import torch.nn.functional as F
from torch_geometric.nn import GCNConv
from torch_geometric.data import Data
from torch_geometric.utils import negative_sampling
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split

d:\Bazar\Travail Yann\CS\3A\MLNS\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Chargement des données

In [2]:
train = pd.read_csv("data/train.txt", sep=" ", header=None)
train.columns = ["u", "v", "label"]

# Graphe de message-passing : uniquement les liens positifs
G = nx.Graph()
pos_edges = train[train["label"] == 1][["u", "v"]].values
G.add_edges_from(pos_edges)
print(f"Noeuds : {G.number_of_nodes()} | Arêtes positives : {G.number_of_edges()}")

Noeuds : 3597 | Arêtes positives : 5248


In [3]:
test = pd.read_csv("data/test.txt", sep=" ", header=None)
test.columns = ["u", "v"]
print(f"Paires à prédire : {len(test)}")

Paires à prédire : 3498


In [4]:
node_info = pd.read_csv("data/node_information.csv", header=None)
node_info = node_info.rename(columns={0: "node"})
node_features = {
    int(row["node"]): row.drop("node").values.astype(np.float32)
    for _, row in node_info.iterrows()
}
feature_dim = len(next(iter(node_features.values())))
print(f"Dimension des features : {feature_dim}")

Dimension des features : 932


## 2. Construction de l'objet PyG Data

In [5]:
# Remappe les nœuds vers des indices contigus 0..N-1
all_nodes = sorted(G.nodes())
node_to_idx = {n: i for i, n in enumerate(all_nodes)}
num_nodes = len(all_nodes)

# Matrice de features (zéros si un nœud n'a pas de feature)
x = torch.zeros((num_nodes, feature_dim), dtype=torch.float)
for node, idx in node_to_idx.items():
    if node in node_features:
        x[idx] = torch.tensor(node_features[node])

# Edge index pour la propagation (graphe non-orienté → doublon)
edge_list = [
    (node_to_idx[u], node_to_idx[v])
    for u, v in G.edges()
    if u in node_to_idx and v in node_to_idx
]
edge_index = torch.tensor(edge_list, dtype=torch.long).t().contiguous()
edge_index = torch.cat([edge_index, edge_index.flip(0)], dim=1)  # non-orienté

data = Data(x=x, edge_index=edge_index, num_nodes=num_nodes)
print(data)

Data(x=[3597, 932], edge_index=[2, 10496], num_nodes=3597)


## 3. Préparation des paires d'entraînement / validation

In [6]:
def pairs_to_tensors(df_pairs, node_to_idx):
    """Convertit un DataFrame (u, v, label) en tenseurs d'indices et de labels."""
    valid = df_pairs[
        df_pairs["u"].isin(node_to_idx) & df_pairs["v"].isin(node_to_idx)
    ].copy()
    u_idx = torch.tensor([node_to_idx[u] for u in valid["u"]], dtype=torch.long)
    v_idx = torch.tensor([node_to_idx[v] for v in valid["v"]], dtype=torch.long)
    labels = torch.tensor(valid["label"].values, dtype=torch.float)
    return u_idx, v_idx, labels

# Split 80/20
train_df, val_df = train_test_split(train, test_size=0.2, stratify=train["label"], random_state=42)
train_u, train_v, train_y = pairs_to_tensors(train_df, node_to_idx)
val_u, val_v, val_y     = pairs_to_tensors(val_df, node_to_idx)

print(f"Train : {len(train_y)} paires | Val : {len(val_y)} paires")
print(f"Ratio positifs (train) : {train_y.mean():.2%}")

Train : 8396 paires | Val : 2100 paires
Ratio positifs (train) : 50.00%


## 4. Modèles : GNN encoder + LinkPredictor

In [7]:
class GNN_model(torch.nn.Module):
    """
    Encodeur GCN : produit un embedding par nœud.
    """
    def __init__(self, num_layers, input_size, hidden_size, output_size, dropout=0.3):
        super().__init__()
        self.convs = torch.nn.ModuleList()
        self.convs.append(GCNConv(input_size, hidden_size))
        for _ in range(num_layers - 2):
            self.convs.append(GCNConv(hidden_size, hidden_size))
        self.convs.append(GCNConv(hidden_size, output_size))
        self.dropout = dropout

    def forward(self, x, edge_index):
        for conv in self.convs[:-1]:
            x = conv(x, edge_index)
            x = F.relu(x)
            # Correction : passage de p et training
            x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.convs[-1](x, edge_index)
        # Pas de softmax : on veut des embeddings bruts pour la prédiction de lien
        return x


class LinkPredictor(torch.nn.Module):
    """
    MLP qui prédit un score à partir des embeddings de deux nœuds.
    """
    def __init__(self, emb_dim, hidden_dim=64):
        super().__init__()
        self.net = torch.nn.Sequential(
            torch.nn.Linear(emb_dim * 2, hidden_dim),
            torch.nn.ReLU(),
            torch.nn.Dropout(0.2),
            torch.nn.Linear(hidden_dim, 1),
        )

    def forward(self, z_u, z_v):
        """Retourne un logit (avant sigmoid) pour chaque paire."""
        return self.net(torch.cat([z_u, z_v], dim=-1)).squeeze(-1)

## 5. Entraînement

In [8]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device : {device}")

# Hyperparamètres
NUM_LAYERS  = 3
HIDDEN_SIZE = 128
EMB_SIZE    = 64
DROPOUT     = 0.3
LR          = 1e-3
EPOCHS      = 500

# Initialisation
data = data.to(device)
train_u, train_v, train_y = train_u.to(device), train_v.to(device), train_y.to(device)
val_u,   val_v,   val_y   = val_u.to(device),   val_v.to(device),   val_y.to(device)

gnn       = GNN_model(NUM_LAYERS, feature_dim, HIDDEN_SIZE, EMB_SIZE, DROPOUT).to(device)
predictor = LinkPredictor(EMB_SIZE, HIDDEN_SIZE).to(device)
optimizer = torch.optim.Adam(
    list(gnn.parameters()) + list(predictor.parameters()), lr=LR
)
# Poids de classe pour compenser le déséquilibre positif/négatif
pos_weight = torch.tensor([(train_y == 0).sum() / (train_y == 1).sum()]).to(device)


def evaluate(z, u_idx, v_idx, y):
    """Calcule la loss et l'AUC sur un split."""
    with torch.no_grad():
        scores = predictor(z[u_idx], z[v_idx])
        loss = F.binary_cross_entropy_with_logits(scores, y, pos_weight=pos_weight)
        probs = torch.sigmoid(scores).cpu().numpy()
        auc = roc_auc_score(y.cpu().numpy(), probs)
    return loss.item(), auc


best_val_auc = 0.0
best_state   = None

for epoch in range(1, EPOCHS + 1):
    gnn.train()
    predictor.train()
    optimizer.zero_grad()

    z = gnn(data.x, data.edge_index)
    scores = predictor(z[train_u], z[train_v])
    loss = F.binary_cross_entropy_with_logits(scores, train_y, pos_weight=pos_weight)
    loss.backward()
    optimizer.step()

    if epoch % 20 == 0:
        gnn.eval()
        predictor.eval()
        z_eval = gnn(data.x, data.edge_index)
        train_loss, train_auc = evaluate(z_eval, train_u, train_v, train_y)
        val_loss,   val_auc   = evaluate(z_eval, val_u,   val_v,   val_y)
        print(
            f"Epoch {epoch:>3} | "
            f"Train loss {train_loss:.4f} AUC {train_auc:.4f} | "
            f"Val loss {val_loss:.4f} AUC {val_auc:.4f}"
        )
        if val_auc > best_val_auc:
            best_val_auc = val_auc
            best_state = {
                "gnn":       {k: v.cpu() for k, v in gnn.state_dict().items()},
                "predictor": {k: v.cpu() for k, v in predictor.state_dict().items()},
            }

print(f"\nMeilleur AUC validation : {best_val_auc:.4f}")

Device : cpu
Epoch  20 | Train loss 0.6374 AUC 0.7064 | Val loss 0.6349 AUC 0.6915
Epoch  40 | Train loss 0.5765 AUC 0.7828 | Val loss 0.6053 AUC 0.7160
Epoch  60 | Train loss 0.4950 AUC 0.8449 | Val loss 0.6092 AUC 0.7318
Epoch  80 | Train loss 0.4248 AUC 0.8908 | Val loss 0.6040 AUC 0.7604
Epoch 100 | Train loss 0.3688 AUC 0.9211 | Val loss 0.6268 AUC 0.7780
Epoch 120 | Train loss 0.3175 AUC 0.9438 | Val loss 0.6333 AUC 0.7895
Epoch 140 | Train loss 0.2642 AUC 0.9615 | Val loss 0.6077 AUC 0.8102
Epoch 160 | Train loss 0.2136 AUC 0.9745 | Val loss 0.5997 AUC 0.8323
Epoch 180 | Train loss 0.1786 AUC 0.9821 | Val loss 0.6307 AUC 0.8433
Epoch 200 | Train loss 0.1537 AUC 0.9869 | Val loss 0.6493 AUC 0.8515
Epoch 220 | Train loss 0.1328 AUC 0.9901 | Val loss 0.6738 AUC 0.8536
Epoch 240 | Train loss 0.1179 AUC 0.9922 | Val loss 0.7132 AUC 0.8552
Epoch 260 | Train loss 0.1061 AUC 0.9935 | Val loss 0.7383 AUC 0.8578
Epoch 280 | Train loss 0.0942 AUC 0.9950 | Val loss 0.7484 AUC 0.8608
Epoch 3

## 6. Inférence sur le test set

In [11]:
# Rechargement du meilleur checkpoint
gnn.load_state_dict({k: v.to(device) for k, v in best_state["gnn"].items()})
predictor.load_state_dict({k: v.to(device) for k, v in best_state["predictor"].items()})
gnn.eval()
predictor.eval()

with torch.no_grad():
    z = gnn(data.x, data.edge_index)

scores = []
for _, row in test.iterrows():
    u, v = int(row["u"]), int(row["v"])
    if u in node_to_idx and v in node_to_idx:
        s = predictor(
            z[node_to_idx[u]].unsqueeze(0),
            z[node_to_idx[v]].unsqueeze(0)
        )
        prob = torch.sigmoid(s).item()
    else:
        # Nœud hors graphe : score de base (degré 0 → peu probable)
        prob = 0.0
    scores.append(prob)

test["score"] = scores
test["predicted_label"] = (test["score"] > 0.5).astype(int)

submit = pd.DataFrame({"ID" : range(len(list(test["predicted_label"]))), "Predicted" : list(test["predicted_label"])})


submit.to_csv("predictions.csv", index=False)
print(test.head(10))
print(f"\nLiens prédits positifs : {test['predicted_label'].sum()} / {len(test)}")

      u     v     score  predicted_label
0  3425  4524  0.000014                0
1  1620  2617  0.787646                1
2  4832  6317  0.000003                0
3  4984  7298  0.981377                1
4   385  5481  0.415091                0
5  1722  2930  0.002679                0
6  1534  3330  0.006864                0
7  5015  6354  0.009319                0
8   856  2504  0.000010                0
9   851  5515  0.014092                0

Liens prédits positifs : 1372 / 3498


## Annexe : ajout optionnel des poids d'arêtes

Le code ci-dessous calcule les poids à partir de la similarité cosinus des features de nœuds.  
Pour les utiliser dans le GCNConv, passer `edge_weight` en argument dans `forward`.

In [10]:
# Calcul des poids d'arêtes (optionnel)
edge_weights = []
for u, v in G.edges():
    x_u = node_features.get(u, np.zeros(feature_dim))
    x_v = node_features.get(v, np.zeros(feature_dim))
    norm = (np.linalg.norm(x_u) * np.linalg.norm(x_v)) + 1e-9
    cosine_sim = np.dot(x_u, x_v) / norm  # cosinus plutôt que dot brut
    edge_weights.append(1.0 + 0.1 * cosine_sim)

# Pour un graphe non-orienté, dupliquer les poids
edge_weight_tensor = torch.tensor(edge_weights * 2, dtype=torch.float).to(device)
print(f"Edge weights : min={edge_weight_tensor.min():.3f}  max={edge_weight_tensor.max():.3f}")

Edge weights : min=1.000  max=1.100
